# 01 - Setup da Landing Zone 01_raw

## Objetivo

Este notebook prepara a estrutura inicial do projeto no Databricks, criando a organização no Unity Catalog para o Lakehouse do dataset Olist.

A camada `01_raw` será usada como Landing Zone, ou seja, o local onde os arquivos CSV originais serão armazenados antes da criação das tabelas Bronze, Silver e Gold.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS tp1_olist;

## 1. Criação dos schemas do projeto

Os schemas representam as camadas lógicas da arquitetura medalhão:

- `01_raw`: arquivos CSV brutos
- `02_bronze`: tabelas brutas em Delta/DLT
- `03_silver`: tabelas tratadas e validadas
- `04_gold`: tabela analítica de negócio
- `05_quarantine`: registros inválidos enviados para quarentena

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS tp1_olist.`01_raw`;
CREATE SCHEMA IF NOT EXISTS tp1_olist.`02_bronze`;
CREATE SCHEMA IF NOT EXISTS tp1_olist.`03_silver`;
CREATE SCHEMA IF NOT EXISTS tp1_olist.`04_gold`;
CREATE SCHEMA IF NOT EXISTS tp1_olist.`05_quarantine`;

## 2. Criação do volume da Landing Zone

Caminho esperado:

`/Volumes/tp1_olist/01_raw/olist_raw/`

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS tp1_olist.`01_raw`.olist_raw;

## 3. Definição do caminho da Landing Zone


In [0]:
raw_path = "/Volumes/tp1_olist/01_raw/olist_raw/"

print(f"Caminho da Landing Zone 01_raw: {raw_path}")

In [0]:
display(dbutils.fs.ls(raw_path))

## 5. Validação dos arquivos da camada 01_raw

Após realizar o upload dos arquivos CSV para a camada `01_raw`, validamos se todos os arquivos obrigatórios do projeto estão disponíveis e acessíveis para o pipeline.

In [0]:
files = dbutils.fs.ls(raw_path)

display(files)

In [0]:
expected_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_payments_dataset.csv",
    "product_category_name_translation.csv"
]

existing_files = [file.name for file in dbutils.fs.ls(raw_path)]

missing_files = [file for file in expected_files if file not in existing_files]

if missing_files:
    print("Arquivos faltando na camada 01_raw:")
    for file in missing_files:
        print(f"- {file}")
else:
    print("Todos os arquivos obrigatórios foram encontrados na camada 01_raw.")

In [0]:
sample_file = f"{raw_path}olist_orders_dataset.csv"

df_orders_sample = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(sample_file)
)

display(df_orders_sample.limit(10))

In [0]:
orders_count = df_orders_sample.count()

print(f"Quantidade de registros em olist_orders_dataset.csv: {orders_count}")

In [0]:
for file_name in expected_files:
    file_path = f"{raw_path}{file_name}"
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )
    
    print(f"{file_name}: {df.count()} registros")